# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# # Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
# openai = OpenAI()




### GEMINI




import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    print("No Gemini API key was found - please check your .env file!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't look like a valid Google key (should start with AQ.); please check your key!")
else:
    print("Gemini API key found and looks good so far!")

Gemini API key found and looks good so far!


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [9]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [12]:
# def select_relevant_links(url):
#     response = openai.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": link_system_prompt},
#             {"role": "user", "content": get_links_user_prompt(url)}
#         ],
#         response_format={"type": "json_object"}
#     )
#     result = response.choices[0].message.content
#     links = json.loads(result)
#     return links
    
    
    
# Practicing with Gemini API call

from google import genai
from google.genai import types

def select_relevant_links(url):
    client = genai.Client()
    
    config = types.GenerateContentConfig(
        system_instruction=link_system_prompt,
        response_mime_type="application/json"
    )
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[get_links_user_prompt(url)],
        config=config
    )
    
    import json
    links = json.loads(response.text)
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'professional profile',
   'url': 'https://edwarddonner.com/curriculum/'}]}

In [14]:
# def select_relevant_links(url):
#     print(f"Selecting relevant links for {url} by calling {MODEL}")
#     response = openai.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": link_system_prompt},
#             {"role": "user", "content": get_links_user_prompt(url)}
#         ],
#         response_format={"type": "json_object"}
#     )
#     result = response.choices[0].message.content
#     links = json.loads(result)
#     print(f"Found {len(links['links'])} relevant links")
#     return links



### GEMINI


from google import genai
from google.genai import types
import json

def select_relevant_links(url):

    model_name = "gemini-2.5-flash" 
    print(f"Selecting relevant links for {url} by calling {model_name}")
    
    client = genai.Client()
    
    config = types.GenerateContentConfig(
        system_instruction=link_system_prompt,
        response_mime_type="application/json"
    )
    
    response = client.models.generate_content(
        model=model_name,
        contents=[get_links_user_prompt(url)],
        config=config
    )
    
    links = json.loads(response.text)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash
Found 8 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'related company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [21]:
print(fetch_page_and_all_relevant_links("https://wikipedia.com"))

Selecting relevant links for https://wikipedia.com by calling gemini-2.5-flash
Found 0 relevant links
## Landing Page:

Wikipedia

Wikipedia
The Free Encyclopedia
English
7,189,000+
articles
日本語
1,503,000+
記事
Deutsch
3.125.000+
Artikel
Русский
2 103 000+
статей
Français
2 761 000+
articles
Español
2.116.000+
artículos
中文
1,537,000+
条目 / 條目
Italiano
1.971.000+
voci
Polski
1 696 000+
haseł
Português
1.173.000+
artigos
Search Wikipedia
Afrikaans
Shqip
العربية
Asturianu
Azərbaycanca
Български
閩南語 / Bân-lâm-gú
বাংলা
Беларуская
Català
Čeština
Cymraeg
Dansk
Deutsch
Eesti
Ελληνικά
English
Español
Esperanto
Euskara
فارسی
Français
Galego
한국어
Hausa
Հայերեն
हिन्दी
Hrvatski
Bahasa Indonesia
Italiano
עברית
ქართული
Ladin
Latina
Latviešu
Lietuvių
Magyar
Македонски
Malagasy
मराठी
مصرى
Bahasa Melayu
Bahaso Minangkabau
မြန်မာဘာသာ
Nederlands
日本語
Norsk (bokmål)
Norsk (nynorsk)
Нохчийн
Oʻzbekcha / Ўзбекча
Polski
Português
Қазақша / Qazaqşa / قازاقشا
Română
Simple English
Sinugboanong Binisaya
Slovenčina
Slo

In [22]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("Wikipedia", "https://wikipedia.com")

Selecting relevant links for https://wikipedia.com by calling gemini-2.5-flash
Found 7 relevant links


'\nYou are looking at a company called: Wikipedia\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nWikipedia\n\nWikipedia\nThe Free Encyclopedia\nEnglish\n7,189,000+\narticles\n日本語\n1,503,000+\n記事\nDeutsch\n3.125.000+\nArtikel\nРусский\n2\xa0103\xa0000+\nстатей\nFrançais\n2\u202f761\u202f000+\narticles\nEspañol\n2.116.000+\nartículos\n中文\n1,537,000+\n条目 / 條目\nItaliano\n1.971.000+\nvoci\nPolski\n1\xa0696\xa0000+\nhaseł\nPortuguês\n1.173.000+\nartigos\nSearch Wikipedia\nAfrikaans\nShqip\nالعربية\nAsturianu\nAzərbaycanca\nБългарски\n閩南語 / Bân-lâm-gú\nবাংলা\nБеларуская\nCatalà\nČeština\nCymraeg\nDansk\nDeutsch\nEesti\nΕλληνικά\nEnglish\nEspañol\nEsperanto\nEuskara\nفارسی\nFrançais\nGalego\n한국어\nHausa\nՀայերեն\nहिन्दी\nHrvatski\nBahasa Indonesia\nItaliano\nעברית\nქართული\nLadin\nLatina\nLatviešu\nLietuvių\nMagyar\nМакедонски\nMalagasy\nमराठी\nمصرى\nBah

In [27]:
# def create_brochure(company_name, url):
#     response = openai.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {"role": "system", "content": brochure_system_prompt},
#             {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
#         ],
#     )
#     result = response.choices[0].message.content
#     display(Markdown(result))



### GEMINI API CALL 

from google import genai
from IPython.display import Markdown, display

def create_brochure(company_name, url):
    client = genai.Client()
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[get_brochure_user_prompt(company_name, url)],
        config={"system_instruction": brochure_system_prompt}
    )
    
    display(Markdown(response.text))

In [28]:
create_brochure("Wikipedia", "https://wikipedia.com")

Selecting relevant links for https://wikipedia.com by calling gemini-2.5-flash
Found 1 relevant links


**Wikipedia: The Free Encyclopedia**

Welcome to Wikipedia, the world's largest and most popular free encyclopedia. We are a global, collaborative project dedicated to making the sum of all human knowledge freely available to everyone, everywhere.

**Our Vision**
To create and distribute a free, accessible, and editable encyclopedia in every language, built and maintained by a vibrant global community. We believe in the power of open knowledge to empower individuals and advance understanding across cultures.

**Our Reach and Impact**
Wikipedia boasts an incredible scale, with:
*   Over 7 million articles in English alone
*   Millions more articles across hundreds of languages, including over 1.5 million in Japanese, German, and Chinese, and over 2 million in French, Russian, and Spanish.
*   A vast linguistic diversity, offering content in languages ranging from Afrikaans to Zulu, ensuring knowledge is accessible to diverse populations worldwide.

**Our Community and Collaboration**
Wikipedia is a testament to the power of global cooperation. It operates as a "global community site for the Wikimedia Foundation's projects," facilitating coordination, documentation, planning, and analysis among contributors. Our content is created, maintained, and curated by millions of volunteers and contributors around the world, making it a truly decentralized and community-driven endeavor. We foster discussions and collaboration through various channels, including mailing lists and forums.

**The Wikimedia Ecosystem**
Wikipedia is a flagship project of the Wikimedia Foundation, a non-profit organization that supports a broader ecosystem of free knowledge projects. These include Wikimedia Commons (a media repository), Wiktionary (a dictionary), Wikiquote, Wikisource, Wikibooks, Wikiversity, Wikinews, Wikidata, MediaWiki (our underlying software), and more. This family of projects works together to spread free knowledge globally.

**For Prospective Customers (Users)**
Anyone with an internet connection is a "customer" of Wikipedia. We provide a vast, multilingual, and constantly evolving source of information for students, researchers, professionals, and curious minds around the globe—all completely free of charge.

**For Investors and Donors**
Supporting Wikipedia means investing in the future of free knowledge. Your contributions help maintain our servers, develop our technology, and support the global community that makes Wikipedia possible. You empower a non-profit mission to provide unbiased, verifiable information to billions, bridging knowledge gaps and fostering global understanding.

**For Recruits and Contributors**
While we don't have specific job listings here, joining the Wikipedia movement means becoming part of a passionate, global community dedicated to a shared mission. Whether you contribute by writing articles, editing, translating, or supporting the underlying infrastructure, you play a vital role in shaping the world's most comprehensive knowledge resource. We welcome anyone eager to contribute to the vision of universal access to knowledge.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>